# Dayli Calendar Playground

Use this notebook to explore the core Dayli product loop:

- take a natural-language planning request
- turn it into structured schedule changes
- test iterative edits like `move my gym to tomorrow`
- inspect the preview before wiring in real calendar APIs


## What this notebook helps you learn

This notebook is meant to make the architecture concrete.

- `ConversationManager` orchestrates the workflow
- `PlannerAgent` decides the kind of request
- `ParserAgent` turns text into structured events
- `ValidatorAgent` applies business rules
- `CalendarAgent` produces a preview or an apply-ready change set

For now, the calendar provider is still stubbed, so this is a safe place to experiment.

In [1]:
from __future__ import annotations

import asyncio
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Using project root: {ROOT}")

Using project root: /Users/bitbase/workspace/github.com/rameezshafat/dayli


In [2]:
from dayli.api.schemas.chat import ChatRequest
from dayli.core.config import get_settings
from dayli.orchestration.conversation_manager import ConversationManager

settings = get_settings()
manager = ConversationManager.bootstrap(settings)
settings

Settings(app_name='Dayli', environment='development', api_prefix='/v1', default_timezone='UTC', openai_model='gpt-4.1-mini', cors_allow_origins=['*'])

## Helper function

This wrapper makes it easy to simulate a user turn and inspect the preview response.

In [3]:
async def run_turn(message: str, mode: str = "preview", session_id: str = "notebook-session"):
    payload = ChatRequest(
        user_id="notebook-user",
        session_id=session_id,
        message=message,
        mode=mode,
    )
    response = await manager.handle_message(payload)
    return {
        "reply": response.reply,
        "changes": {
            "mode": response.changes.mode,
            "events": [event.__dict__ for event in response.changes.events],
            "warnings": response.changes.warnings,
        },
    }


def preview(message: str, mode: str = "preview", session_id: str = "notebook-session"):
    result = asyncio.run(run_turn(message=message, mode=mode, session_id=session_id))
    print(json.dumps(result, indent=2))
    return result

## Try the main Dayli use case

Start with the primary product moment: a user describes their day in natural language.

In [4]:
preview("I want to work from 9-12, gym at 6, dinner at 8")

RuntimeError: asyncio.run() cannot be called from a running event loop

## Try iterative edits

These are important to the product because users will refine their schedule conversationally.

In [5]:
preview("Move my gym to tomorrow")

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
preview("Make my morning less busy")

## Inspect the lower-level parser output

Sometimes it helps to look beneath the final response and inspect what the parser thinks the user asked for.

In [6]:
context = asyncio.run(manager._memory.load("notebook-user", "parser-session"))
plan = asyncio.run(manager._planner.plan("work from 9-12, gym at 6, dinner at 8", context))
parsed = asyncio.run(manager._parser.parse("work from 9-12, gym at 6, dinner at 8", plan, context))

print(plan)
print(parsed)

RuntimeError: asyncio.run() cannot be called from a running event loop

## Try apply mode later

Right now the provider is a stub, so `apply` is safe. Once you connect Google Calendar, treat `apply` as a real write path.

In [ ]:
preview("Move my gym to tomorrow", mode="apply", session_id="apply-session")

## What to explore next

Good experiments for the next version:

- improve the parser for relative dates and times
- add real conflict checking against existing events
- replace the stub provider with Google Calendar OAuth + event writes
- save session memory in Redis/Postgres instead of in-memory placeholders
- test vague prompts like `make the afternoon more relaxed`